### Initialize instruments

In [1]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, LeCroyScope
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
from DataProcessing import Readout

import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from DataProcessing.StarkCalibrate import symmetry_search
# from scipy.signal import find_peaks

from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'
rcParams['mathtext.fontset'] = 'stix'
rcParams['axes.unicode_minus'] = False
rcParams.update({
	'font.size': 14,
	'axes.titlesize': 14,
	'axes.labelsize': 14,
	'xtick.labelsize': 14,
	'ytick.labelsize': 14,
	'legend.fontsize': 14,
	'figure.titlesize': 14,
	'legend.frameon': False,
})

In [ ]:
# set up readout and plotting
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [ ]:
# open list of available instruments
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
# connect to BNC 575 pulse generator (for controlling experimental time sequence)
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
# connect to SR400 gated photon counter (for performing Rydberg atom counting)
sr400 = SR400.SR400("GPIB0::23::INSTR", timeout = 5000)

In [2]:
# connect to WeederTech WTMCD-M driver (for 480 stepper motor)
weeder = Weeder.Weeder("ASRL1::INSTR")

In [ ]:
# connect to MC USB-3114 (for compensation voltages)
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

### Define timing sequence, set up photon counter

In [ ]:
# define timing sequence (MODIFIED MODE); MOT is always on
pulse_arrangement = [         
                     # ["C", 15, 0, 6, 1],   # 480 EXC,
                        ["A", 10e-3,0,2,1]
                    ]
T = 20e-3
notes = {}

In [ ]:
# define timing sequence (STANDARD MODE)
pulse_arrangement = [["A", 1300e-6, 0, 2, 1],                  # MOT AOM
                     [["B", 15e-6, 40e-6, 2, 1],                # 780 AOM
                     ["C", 15e-6, 40e-6, 6, -1]],               # 480 switch
                     ["D", 26e-6, 0, 8, 1],                   # DEI switch/gated photon counter
                    ]

# define sequence period
T = 1500e-6

In [ ]:
# set up gated photon counter
cycle_number = 30

# photon counter gate width and delay
gate_width = 20e-6
gate_delay = 3e-6

# stepper motor slope (for blue laser)
freq_factor = 129 * 2 # kHz

notes = {"gate_width": gate_width, "gate_delay": gate_delay}

In [ ]:
# set up the photon counter
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 100e-6, count_period_num = cycle_number, 
                  dwell = 0, sourceA = "INPUT1", sourceB = "INPUT2", gate_A_mode = "FIXED",
                  gate_A_delay = gate_delay, gate_A_width = gate_width, gate_B_mode = "CW")

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

### Find the Rydberg resonance

In [4]:
# find current stepper motor position
weeder.position(header = "A", query = True)

'A5500'

In [5]:
# begin stepper motor at initial position 6000, then move it out to increase the laser frequency
weeder.move("A", position = 5400, progress = True)

start at 5500 --> 5400.0 --> 5400
Old position: 5500
New position: 5401


In [6]:
# used to move motor incrementally to zero in on the location of a resonance
weeder.advance(header = "A", num_step = -10)

start at 5401 --> 5391.0 --> 5391
Old position: 5401
New position: 5391


### Stark maps and field zeroing

#### 1D map (frequency scan only)

In [ ]:
# set up scanning parameters
center = 7500
stride = 2
half_width = 400

# scan list gneration
x_list = np.arange(2* half_width/stride)

In [ ]:
# scan over the blue laser frequency, record photon counts (averaged over 30 shots)

# when scan begins, we overshoot on the initial position and then move motor to real start position to release the band tension and gear margin
start_position = center - half_width
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

results = []
# s = time.perf_counter()
for idx in x_list:
    print(f"Now checking step: {idx}", end = "\r", flush = True)
    sr400.count_reset()
    result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
    results.append(result)
    for i in range(stride):
        weeder.step("A", "+")
    time.sleep(0.1)
    
# e = time.perf_counter()
# print('Time elapsed: {} seconds'.format(e-s))

In [ ]:
# plot and save results
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results,
          xlabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

#### 2D maps (voltage scan inside of frequency scan; small-voltage range)

#### 2D maps (voltage scan inside of frequency scan; wide voltage range) and field zeroing

In [ ]:
center = center  # center point of frequency scan
stride = 2  # step size
half_width = 300  # step range

# scan list gneration
s_list = np.arange(2*half_width/stride)

# voltage setpoints
#vset = np.array([0.2,-0.1,0.6])
#vset = np.array([0.2,-0.2,0.4])
vset = np.array([0.0, 0.0, 0.0]) # set all elements to zero when starting the field zeroing, update as new Stark maps are generated 

num_voltages = 31  # number of voltages to test

# voltage arrays for each axis
vx_list = np.linspace(-2.0, 2.0, num_voltages) + vset[0]
vy_list = np.linspace(-2.0, 2.0, num_voltages) + vset[1]
vz_list = np.linspace(-2.0, 2.0, num_voltages) + vset[2]

v_list = np.vstack((vx_list, vy_list, vz_list))

# MC channels for each electrode pair
electrode_pair = [[10,11],[6,7],[0,1]]

In [ ]:
# initialize electrode voltages
mcbox.set_voltage_1chan(channel = 10, voltage = vset[0], bipolar = True, display = True)
mcbox.set_voltage_1chan(channel = 11, voltage = -vset[0], bipolar = True, display = True)

mcbox.set_voltage_1chan(channel = 6, voltage = vset[1], bipolar = True, display = True)
mcbox.set_voltage_1chan(channel = 7, voltage = -vset[1], bipolar = True, display = True)

mcbox.set_voltage_1chan(channel = 0, voltage = 0, bipolar = True, display = True)
mcbox.set_voltage_1chan(channel = 1, voltage = -vset[2], bipolar = True, display = True)

In [ ]:
# voltage scan inside a frequency scan, collect counts from gated counter
start_position = center - half_width

stark_map = []

# electrode pair to check; i = 0,1,2 for x,y,z
i = 0

test_voltages = generate_test_voltages(v_list[i])
print('Voltages to test: \n {}'.format(test_voltages))

print('Now checking axis {}'.format(i))
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

# s = time.perf_counter()
# print('start:',s)

for s in s_list:
    for j in range(stride):
            weeder.step("A", "+")
    print(f"Now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in test_voltages:
        # scan voltage
        if electrode_pair[i][0] == 0:
            mcbox.set_voltage_1chan(channel = electrode_pair[i][0], voltage = 0, bipolar = True, display = False)
        else:
            mcbox.set_voltage_1chan(channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display = False)
        mcbox.set_voltage_1chan(channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display = False)
        time.sleep(0.01)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
        results.append(result)
    
    stark_map.append(results)

# e = time.perf_counter()
# print('end:',e)
# print('Time elapsed: {} seconds'.format(e-s))

In [ ]:
# plot results
ps.Plot3D(x = test_voltages, y = s_list*stride*freq_factor, Z = np.array(stark_map).T,
          xlabel = f"Voltage {i} (V)", ylabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

In [ ]:
# perform field zeroing
date = "2026/08/12"
plot_id = 1
symmetry_search(date, plot_id, axis = i, plot_best = False, plot_all = False)

### Shut down instrumentation

In [ ]:
# reset voltages to zero
vset = np.array([0.0,0.0,0.0])
mcbox.set_voltage_Nchan(channels = [0,1,6,7,10,11], voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# save current stepper motor position
weeder.save(header = "A", query = True)

In [ ]:
# OPTIONAL: disables pulses from BNC box (can also be done by pressing RUN/STOP button or individual channel buttons on device)
bnc575.disable_all()

In [ ]:
# OPTIONAL: disarms the pulse generator (can also be done by pressing RUN/STOP button on device)
bnc575.disarm_all()

In [ ]:
# OPTIONAL: used to reset photon counter (can also be done by pressing STOP button on device)
sr400.count_reset()

In [ ]:
# disconnect from pulse generator
bnc575.pyvisa.close()

In [ ]:
# disconnect from SRS photon counter
sr400.pyvisa.close()

In [ ]:
f_list = np.linspace(80,120,10)*10**6

In [ ]:
t_list = 1/f_list

In [ ]:
2e9/(625e6)

In [ ]:
import math
np.lcm.reduce(t_list)

In [ ]:
import numpy as np

def lcm_float_array_numpy(arr):
    max_decimals = max(len(str(val).split('.')[-1]) if '.' in str(val) else 0 for val in arr)
    multiplier = 10 ** max_decimals
    int_arr = np.array([int(val * multiplier) for val in arr])
    
    # np.lcm.reduce computes the LCM across the array
    int_lcm = np.lcm.reduce(int_arr)
    
    return int_lcm / multiplier

# Example

print(f"LCM: {lcm_float_array_numpy(t_list)}")   

In [ ]:
lcm_float_array_numpy(t_list)/t_list[2]